# LeetCode 3826 — Minimum Partition Score bằng Python

- Link: https://leetcode.com/problems/minimum-partition-score/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động chia mảng thành `k` đoạn, tổng tiền tố, **Convex Hull Trick** (bao lồi đường thẳng, deque đơn điệu), tối ưu **chia để trị** (divide & conquer optimization)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minPartitionScore(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Viết DP chia mảng thành **đúng `k`** đoạn: `f_t[i] = min_j f_{t−1}[j] + cost(j, i)` với `cost` tính bằng tổng tiền tố.
2. Khai triển chi phí `s(s+1)/2` với `s = P_i − P_j` thành dạng **`m_j · x + b_j`** tại `x = P_i` ⇒ mỗi `j` là một **đường thẳng**, bài toán con là "min các đường thẳng tại một điểm".
3. Cài đặt **Convex Hull Trick** với độ dốc giảm dần và truy vấn tăng dần bằng **deque** — `O(n)` mỗi tầng.
4. Biết **tối ưu chia để trị** dựa trên tính đơn điệu của điểm cắt tối ưu.
5. Kiểm thử với vét cạn mọi cách cắt.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Chia `nums` thành **đúng `k`** đoạn con liên tiếp, khác rỗng. Giá trị của một đoạn có tổng `s` là `s(s+1)/2`. Điểm của cách chia = tổng giá trị các đoạn. Tìm điểm nhỏ nhất.

**Ví dụ 1:** `nums = [5,1,2,1], k = 2` → `25` (`[5]` → 15, `[1,2,1]` → 10).

**Ví dụ 2:** `nums = [1,2,3,4], k = 1` → `55`.

**Ví dụ 3:** `nums = [1,1,1], k = 3` → `3`.

**Ràng buộc:** `1 ≤ n ≤ 1000`, `1 ≤ nums[i] ≤ 10^4`, `1 ≤ k ≤ n`.

In [1]:
from typing import List
from itertools import combinations
from collections import deque
import random, time

INF = float('inf')
examples = [([5, 1, 2, 1], 2, 25), ([1, 2, 3, 4], 1, 55), ([1, 1, 1], 3, 3)]
for nums, k, exp in examples:
    print(nums, "k =", k, "-> kỳ vọng", exp)

[5, 1, 2, 1] k = 2 -> kỳ vọng 25
[1, 2, 3, 4] k = 1 -> kỳ vọng 55
[1, 1, 1] k = 3 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Tổng tiền tố và chi phí đoạn

`P[0] = 0`, `P[i] = nums[0] + … + nums[i−1]`. Đoạn `nums[j..i−1]` có tổng `s = P[i] − P[j]`, chi phí `s(s+1)/2`. Vì `nums[i] ≥ 1`, `P` **tăng ngặt**.

In [2]:
nums = [5, 1, 2, 1]
P = [0]
for x in nums: P.append(P[-1] + x)
cost = lambda j, i: (P[i] - P[j]) * (P[i] - P[j] + 1) // 2
print("P =", P, "| cost([5]) =", cost(0, 1), "| cost([1,2,1]) =", cost(1, 4))

P = [0, 5, 6, 8, 9] | cost([5]) = 15 | cost([1,2,1]) = 10


### 3.2. Hàm `s(s+1)/2` lồi ⇒ chia đều có lợi

`g(s) = s(s+1)/2` lồi (bình phương). Với tổng cố định, chia thành nhiều đoạn có tổng gần bằng nhau làm tổng chi phí nhỏ. Ví dụ tổng 8 chia hai: `8 = 4 + 4` ⇒ `10 + 10 = 20`; `8 = 7 + 1` ⇒ `28 + 1 = 29`.

In [3]:
g = lambda s: s * (s + 1) // 2
print([(a, 8 - a, g(a) + g(8 - a)) for a in range(1, 8)])

[(1, 7, 29), (2, 6, 24), (3, 5, 21), (4, 4, 20), (5, 3, 21), (6, 2, 24), (7, 1, 29)]


### 3.3. Min của nhiều đường thẳng — Convex Hull Trick (CHT)

Cho tập đường thẳng `y = m x + b`, truy vấn `min_l l(x)`. Nếu thêm đường theo **độ dốc giảm dần** và truy vấn `x` **tăng dần**, giữ deque các đường "hữu ích":

- **thêm** đường mới vào cuối: bỏ đường áp chót nếu nó không bao giờ là min (giao điểm kiểm tra bằng phép nhân chéo, tránh chia);
- **truy vấn** `x`: bỏ đường đầu trong khi đường thứ hai cho giá trị `≤` tại `x`; đầu deque là đáp án.

Mỗi đường vào/ra deque một lần ⇒ `O(1)` khấu hao.

In [4]:
class MonotoneCHT:
    """min của các đường y = m*x + b; thêm m giảm dần, truy vấn x tăng dần."""
    def __init__(self):
        self.dq = deque()
    @staticmethod
    def bad(l1, l2, l3):
        # l2 thừa nếu giao(l1, l3) nằm bên trái (hoặc tại) giao(l1, l2)
        (m1, b1), (m2, b2), (m3, b3) = l1, l2, l3
        return (b3 - b1) * (m1 - m2) <= (b2 - b1) * (m1 - m3)
    def add(self, m, b):
        dq = self.dq
        while len(dq) >= 2 and self.bad(dq[-2], dq[-1], (m, b)):
            dq.pop()
        dq.append((m, b))
    def query(self, x):
        dq = self.dq
        while len(dq) >= 2 and dq[1][0] * x + dq[1][1] <= dq[0][0] * x + dq[0][1]:
            dq.popleft()
        m, b = dq[0]
        return m * x + b

lines = [(3, 0), (1, 2), (0, 5), (-2, 20)]
cht = MonotoneCHT()
for m, b in lines: cht.add(m, b)
for x in [0, 1, 2, 5, 9]:
    print(x, cht.query(x), min(m * x + b for m, b in lines))

0 0 0
1 3 3
2 4 4
5 5 5
9 2 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `[5, 1, 2, 1]`, `k = 2`. Ba cách cắt:

| cắt | đoạn | tổng | điểm |
|---|---|---|---|
| sau 5 | [5] [1,2,1] | 5, 4 | 15 + 10 = **25** |
| sau 1 | [5,1] [2,1] | 6, 3 | 21 + 6 = 27 |
| sau 2 | [5,1,2] [1] | 8, 1 | 36 + 1 = 37 |

Đúng như trực giác lồi: hai đoạn có tổng gần nhau nhất (`5` và `4`) cho điểm nhỏ nhất.

Tổng quát: `f_t[i]` = điểm nhỏ nhất để chia `nums[0..i−1]` thành `t` đoạn. Đoạn cuối là `nums[j..i−1]`:

`f_t[i] = min_{t−1 ≤ j < i} ( f_{t−1}[j] + g(P_i − P_j) )`.

## 5. Cách 1 — Brute Force: thử mọi vị trí cắt

Chọn `k − 1` vị trí cắt trong `n − 1` khe: `C(n−1, k−1)` cách. Chỉ dùng cho `n` nhỏ.

In [5]:
def brute(nums, k):
    n = len(nums)
    P = [0]
    for x in nums: P.append(P[-1] + x)
    best = INF
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        best = min(best, sum((P[b[t + 1]] - P[b[t]]) * (P[b[t + 1]] - P[b[t]] + 1) // 2 for t in range(k)))
    return best

for nums, k, exp in examples:
    print(brute(nums, k), "kỳ vọng", exp)

25 kỳ vọng 25
55 kỳ vọng 55
3 kỳ vọng 3


**Nút thắt:** `C(999, 499)` khổng lồ. DP gộp theo `(t, i)` còn `O(k n²) = 10^9` — vẫn quá chậm. Cần tăng tốc bước `min_j`.

## 6. Ý tưởng thuật toán / bước đột phá

**Khai triển chi phí.** Với `x = P_i`, `y = P_j`:

`g(x − y) = (x − y)(x − y + 1)/2 = (x² + x)/2 − x·y + (y² − y)/2`.

Vậy

`f_t[i] = (x² + x)/2 + min_j [ (−P_j) · x + ( f_{t−1}[j] + (P_j² − P_j)/2 ) ]`.

Mỗi `j` là **đường thẳng** với độ dốc `m_j = −P_j` và tung độ `b_j = f_{t−1}[j] + (P_j² − P_j)/2`. Phần `min_j` là truy vấn CHT tại `x = P_i`.

**Đơn điệu kép:** `P_j` tăng ⇒ độ dốc **giảm dần** theo `j`; `P_i` tăng ⇒ truy vấn **tăng dần**. ⇒ CHT bằng deque, `O(n)` mỗi tầng, tổng `O(k n)`.

**Thứ tự:** ở tầng `t`, khi tính `f_t[i]` ta chỉ cho phép `j ≤ i − 1` ⇒ thêm đường `j = i − 1` ngay trước truy vấn `i`.

## 7. Các cách tiếp theo

- **Cách 2:** DP thẳng `O(k n²)`.
- **Cách 3:** DP + **Convex Hull Trick** (deque) — `O(k n)` (lời giải nộp chính).
- **Cách 4:** DP + **tối ưu chia để trị**: điểm cắt tối ưu `opt(i)` không giảm theo `i` ⇒ `O(k n log n)`.

### 7.1. Cách 2 — DP `O(k n²)`

In [6]:
def dp_quadratic(nums, k):
    n = len(nums)
    P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)
        for i in range(t, n + 1):
            Pi = P[i]
            best = INF
            for j in range(t - 1, i):
                if prev[j] < INF:
                    s = Pi - P[j]
                    v = prev[j] + s * (s + 1) // 2
                    if v < best: best = v
            cur[i] = best
        prev = cur
    return prev[n]

for nums, k, exp in examples:
    print(dp_quadratic(nums, k), "kỳ vọng", exp)

25 kỳ vọng 25
55 kỳ vọng 55
3 kỳ vọng 3


**Độ phức tạp:** `O(k · n²)` — với `n = k = 1000` là `10^9`, không khả thi trong Python.

### 7.2. Cách 3 — DP + Convex Hull Trick (lời giải chính)

**Giải thích từng đoạn:**

1. `prev = f_{t−1}` (ban đầu `f_0[0] = 0`, còn lại `∞`).
2. Tầng `t`: CHT rỗng. Với `i = t..n`: thêm đường của `j = i − 1` (nếu `prev[j]` hữu hạn), rồi `cur[i] = (P_i² + P_i)/2 + cht.query(P_i)`.
3. Sau `k` tầng, đáp án `prev[n]`.

Mọi phép toán là số nguyên (vì `P_j² − P_j` và `P_i² + P_i` đều chẵn).

In [7]:
def dp_cht(nums, k):
    n = len(nums)
    P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)
        cht = MonotoneCHT()
        for i in range(t, n + 1):
            j = i - 1
            if prev[j] < INF:
                cht.add(-P[j], prev[j] + (P[j] * P[j] - P[j]) // 2)
            x = P[i]
            cur[i] = (x * x + x) // 2 + cht.query(x)
        prev = cur
    return prev[n]

for nums, k, exp in examples:
    print(dp_cht(nums, k), "kỳ vọng", exp)

25 kỳ vọng 25
55 kỳ vọng 55
3 kỳ vọng 3


**Độ phức tạp Cách 3:** mỗi tầng `O(n)` (khấu hao) ⇒ `O(k · n)` ≤ `10^6`; bộ nhớ `O(n)`.

### 7.3. Cách 4 — Tối ưu chia để trị

**Tính chất:** chi phí `w(j, i) = g(P_i − P_j)` thỏa **bất đẳng thức tứ giác** (vì `g` lồi và `P` tăng), nên điểm cắt tối ưu `opt_t(i)` **không giảm** theo `i`. Tính `cur[mid]` bằng cách duyệt `j` trong `[optL, optR]`, rồi đệ quy hai nửa với khoảng `opt` thu hẹp.

In [8]:
def dp_divide_conquer(nums, k):
    n = len(nums)
    P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)
        stack = [(t, n, t - 1, n - 1)]          # (lo, hi, optL, optR) — dùng ngăn xếp thay đệ quy
        while stack:
            lo, hi, oL, oR = stack.pop()
            if lo > hi:
                continue
            mid = (lo + hi) // 2
            best, bj = INF, oL
            for j in range(oL, min(mid - 1, oR) + 1):
                if prev[j] < INF:
                    s = P[mid] - P[j]
                    v = prev[j] + s * (s + 1) // 2
                    if v < best:
                        best, bj = v, j
            cur[mid] = best
            stack.append((lo, mid - 1, oL, bj))
            stack.append((mid + 1, hi, bj, oR))
        prev = cur
    return prev[n]

for nums, k, exp in examples:
    print(dp_divide_conquer(nums, k), "kỳ vọng", exp)

25 kỳ vọng 25
55 kỳ vọng 55
3 kỳ vọng 3


In [9]:
class SolutionDC:
    def minPartitionScore(self, nums: List[int], k: int) -> int:
        return dp_divide_conquer(nums, k)

print([SolutionDC().minPartitionScore(a, k) for a, k, _ in examples])

[25, 55, 3]


**Độ phức tạp Cách 4:** `O(k · n log n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: chỉ cần tính chất đơn điệu của `opt`, không cần khai triển thành đường thẳng.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 (`P = [0, 5, 6, 8, 9]`, `k = 2`). Tầng `t = 1`: `f_1[i] = g(P_i)` = `[∞, 15, 21, 36, 45]`. Tầng `t = 2`: in các đường được thêm và giá trị truy vấn.

In [10]:
def trace(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    print("P =", P)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); cht = MonotoneCHT()
        for i in range(t, n + 1):
            j = i - 1
            if prev[j] < INF:
                line = (-P[j], prev[j] + (P[j] * P[j] - P[j]) // 2)
                cht.add(*line)
            x = P[i]; q = cht.query(x)
            cur[i] = (x * x + x) // 2 + q
            print(f"t={t} i={i}: thêm đường j={j} {line if prev[j] < INF else '(bỏ qua, ∞)'} | deque={list(cht.dq)} | f={cur[i]}")
        prev = cur
    return prev[n]

print("đáp án:", trace([5, 1, 2, 1], 2))

P = [0, 5, 6, 8, 9]
t=1 i=1: thêm đường j=0 (0, 0) | deque=[(0, 0)] | f=15
t=1 i=2: thêm đường j=1 (bỏ qua, ∞) | deque=[(0, 0)] | f=21
t=1 i=3: thêm đường j=2 (bỏ qua, ∞) | deque=[(0, 0)] | f=36
t=1 i=4: thêm đường j=3 (bỏ qua, ∞) | deque=[(0, 0)] | f=45
t=2 i=2: thêm đường j=1 (-5, 25) | deque=[(-5, 25)] | f=16
t=2 i=3: thêm đường j=2 (-6, 36) | deque=[(-5, 25), (-6, 36)] | f=21
t=2 i=4: thêm đường j=3 (-8, 64) | deque=[(-5, 25), (-6, 36), (-8, 64)] | f=25
đáp án: 25


Đọc `t = 2, i = 4`: deque chứa các đường của `j = 1, 2, 3`; giá trị nhỏ nhất tại `x = 9` đến từ `j = 1` (cắt sau `5`): `f_1[1] + g(9 − 5) = 15 + 10 = 25` ✔.

## 9. Code nộp LeetCode

In [11]:
from collections import deque

class Solution:
    def minPartitionScore(self, nums: List[int], k: int) -> int:
        n = len(nums)
        P = [0] * (n + 1)
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x
        INF = float('inf')
        prev = [0] + [INF] * n
        for t in range(1, k + 1):
            cur = [INF] * (n + 1)
            ms, bs = deque(), deque()            # deque đường thẳng (độ dốc giảm dần)
            for i in range(t, n + 1):
                j = i - 1
                if prev[j] < INF:
                    m, b = -P[j], prev[j] + (P[j] * P[j] - P[j]) // 2
                    while len(ms) >= 2 and (b - bs[-2]) * (ms[-2] - ms[-1]) <= (bs[-1] - bs[-2]) * (ms[-2] - m):
                        ms.pop(); bs.pop()
                    ms.append(m); bs.append(b)
                x = P[i]
                while len(ms) >= 2 and ms[1] * x + bs[1] <= ms[0] * x + bs[0]:
                    ms.popleft(); bs.popleft()
                cur[i] = (x * x + x) // 2 + ms[0] * x + bs[0]
            prev = cur
        return prev[n]

sol = Solution()
for nums, k, exp in examples:
    got = sol.minPartitionScore(nums, k)
    print(nums, k, "->", got, "✔" if got == exp else "✘")

[5, 1, 2, 1] 2 -> 25 ✔
[1, 2, 3, 4] 1 -> 55 ✔
[1, 1, 1] 3 -> 3 ✔


## 10. Vì sao thuật toán đúng

1. **DP:** mọi cách chia `nums[0..i−1]` thành `t` đoạn có đoạn cuối `nums[j..i−1]` với `t−1 ≤ j < i`; phần trước là cách chia `t−1` đoạn của `nums[0..j−1]`. Tối ưu phần trước độc lập ⇒ công thức `f_t[i]`.
2. **Khai triển:** `g(x − y) = (x² + x)/2 − xy + (y² − y)/2` là đẳng thức đại số; phần chỉ phụ thuộc `x` tách khỏi `min`.
3. **CHT:** với độ dốc thêm vào **giảm dần**, tập các đường đạt min tạo thành bao dưới; đường giữa `l2` thừa khi giao điểm `l1–l3` nằm bên trái giao `l1–l2` (kiểm tra bằng nhân chéo, các mẫu số dương). Truy vấn tăng dần ⇒ đường đầu deque bị vượt sẽ không bao giờ tốt lại ⇒ bỏ được.
4. **Đúng phạm vi `j`:** ở tầng `t`, đường `j` chỉ được thêm khi `prev[j]` hữu hạn (tức `j ≥ t − 1`), và trước khi truy vấn `i = j + 1` ⇒ mọi `j < i` hợp lệ đều có mặt.
5. **Chia để trị** (Cách 4): `w(a, c) + w(b, d) ≤ w(a, d) + w(b, c)` với `a ≤ b ≤ c ≤ d` (do `g` lồi) ⇒ `opt` đơn điệu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(C(n−1, k−1) · k)` | `O(k)` |
| 2. DP thẳng | `O(k n²)` | `O(n)` |
| 3. DP + CHT | `O(k n)` | `O(n)` |
| 4. DP + chia để trị | `O(k n log n)` | `O(n)` |

Giá trị lớn nhất: `P_n ≤ 10^7` ⇒ `g ≈ 5·10^{13}` — vừa `int64`; Python không lo tràn.

## 12. Bộ test

In [12]:
def run_all(nums, k, with_brute=True):
    res = [dp_cht(nums, k), dp_divide_conquer(nums, k), Solution().minPartitionScore(nums, k)]
    if len(nums) <= 80:
        res.append(dp_quadratic(nums, k))
    if with_brute:
        res.append(brute(nums, k))
    return res

for nums, k, exp in examples:
    assert all(v == exp for v in run_all(nums, k))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [13]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([7], 1, 28),
    ([1, 1, 1, 1], 2, 6),                      # 2 + 2 -> 3 + 3
    ([10**4] * 5, 5, 5 * (10**4 * (10**4 + 1) // 2)),
    ([1, 100, 1], 2, None),
    ([3, 3, 3, 3, 3, 3], 3, 3 * 21),          # mỗi đoạn tổng 6
]
for nums, k, exp in cases:
    r = run_all(nums, k)
    assert len(set(r)) == 1, (nums, k, r)
    if exp is not None:
        assert r[0] == exp, (nums, k, r)
    print(nums, k, "->", r[0])

[7] 1 -> 28
[1, 1, 1, 1] 2 -> 6
[10000, 10000, 10000, 10000, 10000] 5 -> 250025000
[1, 100, 1] 2 -> 5152
[3, 3, 3, 3, 3, 3] 3 -> 63


In [14]:
# 12.3 phản ví dụ: tham lam "cắt để mỗi đoạn có tổng ≈ tổng/k" (cắt tuần tự) không luôn tối ưu
def greedy_equal(nums, k):
    total = sum(nums); target = total / k
    parts, cur = [], 0
    for idx, x in enumerate(nums):
        remaining_elems = len(nums) - idx
        if cur and (cur + x > target) and len(parts) < k - 1:
            parts.append(cur); cur = 0
        cur += x
        if len(nums) - idx - 1 == (k - 1 - len(parts)) and len(parts) < k - 1:
            parts.append(cur); cur = 0
    parts.append(cur)
    if len(parts) != k or 0 in parts: return None
    return sum(s * (s + 1) // 2 for s in parts)

rng = random.Random(1)
for _ in range(5000):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(2, 8))]; k = rng.randint(1, len(nums))
    gq = greedy_equal(nums, k)
    if gq is not None and gq != Solution().minPartitionScore(nums, k):
        print("phản ví dụ:", nums, k, "| tham lam:", gq, "| đúng:", Solution().minPartitionScore(nums, k)); break

phản ví dụ: [8, 8, 7, 4, 2, 8, 1, 7] 7 | tham lam: 186 | đúng: 185


In [15]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3826)
for _ in range(300):
    n = rng.randint(1, 10); nums = [rng.randint(1, 20) for _ in range(n)]; k = rng.randint(1, n)
    r = run_all(nums, k)
    assert len(set(r)) == 1, (nums, k, r)
print("300 test có vét cạn: OK")
for _ in range(30):
    n = rng.randint(20, 80); nums = [rng.randint(1, 10**4) for _ in range(n)]; k = rng.randint(1, n)
    r = run_all(nums, k, with_brute=False)
    assert len(set(r)) == 1
print("30 test vừa (so với DP thẳng): OK")

300 test có vét cạn: OK


30 test vừa (so với DP thẳng): OK


In [16]:
# 12.5 test lớn đúng ràng buộc
rng = random.Random(9)
nums = [rng.randint(1, 10**4) for _ in range(1000)]
for k in (1, 30, 500, 1000):
    t0 = time.time(); a = Solution().minPartitionScore(nums, k); t1 = time.time()
    extra = "" if k > 60 else f", khớp chia để trị: {a == dp_divide_conquer(nums, k)}"
    print(f"k={k}: {a}, thời gian {t1 - t0:.2f}s{extra}")

k=1: 11751127803828, thời gian 0.00s, khớp chia để trị: True


k=30: 391754158590, thời gian 0.02s, khớp chia để trị: True


k=500: 24193656287, thời gian 0.33s


k=1000: 16137737932, thời gian 0.37s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khởi tạo tầng 0 sai (`f_0[j] = 0` với mọi `j`)

`f_0` chỉ được bằng 0 tại `j = 0` (chưa dùng phần tử nào). Nếu đặt `f_0[j] = 0` cho mọi `j`, đoạn đầu tiên có thể "bắt đầu" ở giữa mảng — tức **bỏ qua** các phần tử đầu — cho đáp án nhỏ sai.

In [17]:
def wrong_init(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] * (n + 1)                                       # SAI: phải là [0, inf, inf, ...]
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); cht = MonotoneCHT()
        for i in range(t, n + 1):
            j = i - 1
            cht.add(-P[j], prev[j] + (P[j] ** 2 - P[j]) // 2)
            cur[i] = (P[i] ** 2 + P[i]) // 2 + cht.query(P[i])
        prev = cur
    return prev[n]

print("sai:", wrong_init([5, 1, 2, 1], 2), "| đúng:", Solution().minPartitionScore([5, 1, 2, 1], 2))

sai: 4 | đúng: 25


**Ghi chú thú vị:** một lỗi khác thường gặp — cho phép **đoạn rỗng** (thêm đường `j = i` trước khi truy vấn `i`) — ở bài này lại **vô hại**: vì `g(a + b) − g(a) − g(b) = a·b ≥ 0`, tách một đoạn luôn không làm tăng chi phí, nên dùng ít đoạn thật hơn (kèm đoạn rỗng chi phí 0) không bao giờ tốt hơn. Tuy vậy với hàm chi phí khác (ví dụ có phí cố định mỗi đoạn) lỗi này sẽ lộ ra, nên vẫn cần cài đúng.

In [18]:
g = lambda s: s * (s + 1) // 2
print([(a, b, g(a + b) - g(a) - g(b), a * b) for a, b in [(1, 1), (3, 5), (10, 7)]])

[(1, 1, 1, 1), (3, 5, 15, 15), (10, 7, 70, 70)]


### Lỗi 2 — Điều kiện bỏ đường thừa bị đảo dấu / dùng phép chia số thực

Kiểm tra giao điểm bằng `float` có thể sai vì làm tròn (giá trị tới `10^{13}`); đảo dấu bất đẳng thức làm deque giữ đường sai. Luôn dùng **nhân chéo số nguyên** và kiểm chứng với DP thẳng.

In [19]:
class WrongCHT(MonotoneCHT):
    @staticmethod
    def bad(l1, l2, l3):
        (m1, b1), (m2, b2), (m3, b3) = l1, l2, l3
        return (b3 - b1) * (m1 - m2) >= (b2 - b1) * (m1 - m3)      # SAI: đảo dấu

def dp_wrong_cht(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); cht = WrongCHT()
        for i in range(t, n + 1):
            if prev[i - 1] < INF: cht.add(-P[i - 1], prev[i - 1] + (P[i - 1] ** 2 - P[i - 1]) // 2)
            cur[i] = (P[i] ** 2 + P[i]) // 2 + cht.query(P[i])
        prev = cur
    return prev[n]

rng = random.Random(2)
for _ in range(3000):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(2, 9))]; k = rng.randint(1, len(nums))
    if dp_wrong_cht(nums, k) != Solution().minPartitionScore(nums, k):
        print("phản ví dụ:", nums, k, "| sai:", dp_wrong_cht(nums, k), "| đúng:", Solution().minPartitionScore(nums, k)); break

phản ví dụ: [6, 8, 3, 7, 8, 9, 4] 4 | sai: 308 | đúng: 287


### Lỗi 3 — Quên phần `(x² + x)/2` khi cộng kết quả truy vấn

Truy vấn CHT chỉ trả `min_j (m_j x + b_j)`; phải cộng thêm phần không phụ thuộc `j`.

In [20]:
def dp_forget_term(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); cht = MonotoneCHT()
        for i in range(t, n + 1):
            if prev[i - 1] < INF: cht.add(-P[i - 1], prev[i - 1] + (P[i - 1] ** 2 - P[i - 1]) // 2)
            cur[i] = cht.query(P[i])                           # SAI: thiếu (x^2 + x)/2
        prev = cur
    return prev[n]

print("sai:", dp_forget_term([5, 1, 2, 1], 2), "| đúng:", Solution().minPartitionScore([5, 1, 2, 1], 2))

sai: -44 | đúng: 25


### Lỗi 4 — Dùng DP `O(k n²)` cho `n = k = 1000`

`10^9` phép tính: quá thời gian. Ô dưới ước lượng thời gian bằng cách chạy DP thẳng ở cỡ nhỏ hơn và nhân lên.

In [21]:
nums = [random.Random(3).randint(1, 100) for _ in range(150)]
t0 = time.time(); dp_quadratic(nums, 50); dt = time.time() - t0
print(f"n=150, k=50: {dt:.2f}s -> ước lượng n=k=1000: ~{dt * (1000 / 150) ** 2 * (1000 / 50):.0f}s")

n=150, k=50: 0.04s -> ước lượng n=k=1000: ~40s


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Chia mảng thành đúng `k` đoạn, tối thiểu tổng chi phí đoạn"** ⇒ DP `f_t[i] = min_j f_{t−1}[j] + w(j, i)`.
- `w(j, i)` là **đa thức bậc 2 của tổng tiền tố** ⇒ khai triển thành đường thẳng ⇒ **Convex Hull Trick**.
- Tổng tiền tố **tăng ngặt** (phần tử dương) ⇒ độ dốc và truy vấn đều đơn điệu ⇒ CHT bằng deque.
- `w` thỏa bất đẳng thức tứ giác ⇒ **tối ưu chia để trị** hoặc **Knuth**; nếu `f(k)` lồi theo `k` còn có thể dùng **WQS (Aliens trick)**.

## 15. Biến thể của bài

1. **Không cố định số đoạn**, mỗi đoạn trả thêm phí `C`: một tầng DP + CHT (dạng *Print Article* kinh điển).
2. **Chi phí `s²`** (không có `+ s`): cùng kỹ thuật, khác hằng số.
3. **Phần tử có thể âm**: `P` không còn đơn điệu ⇒ cần CHT tổng quát (Li Chao tree) hoặc deque + tìm nhị phân.
4. **Tối đa hóa** với hàm lõm: đổi dấu.

In [22]:
# Biến thể 1: số đoạn tùy ý, mỗi đoạn trả thêm phí C (một tầng CHT)
def any_parts_with_fee(nums, C):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    f = [0] + [INF] * n; cht = MonotoneCHT()
    for i in range(1, n + 1):
        j = i - 1
        cht.add(-P[j], f[j] + (P[j] ** 2 - P[j]) // 2)
        f[i] = (P[i] ** 2 + P[i]) // 2 + cht.query(P[i]) + C
    return f[n]

def brute_fee(nums, C):
    return min(Solution().minPartitionScore(nums, k) + C * k for k in range(1, len(nums) + 1))

rng = random.Random(15)
for _ in range(200):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 12))]; C = rng.randint(0, 200)
    assert any_parts_with_fee(nums, C) == brute_fee(nums, C)
print("Biến thể 1 khớp (đối chiếu với min_k của lời giải chính + C·k).")

Biến thể 1 khớp (đối chiếu với min_k của lời giải chính + C·k).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi vị trí cắt | tổ hợp | `O(k)` | Kiểm thử |
| 2. DP thẳng | `min_j` duyệt hết | `O(k n²)` | `O(n)` | `n ≤ 200` |
| 3. DP + CHT | Đường thẳng + deque | `O(k n)` | `O(n)` | **Lời giải chuẩn** |
| 4. DP + chia để trị | `opt` đơn điệu | `O(k n log n)` | `O(n)` | Chi phí khó khai triển thành đường thẳng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `minPartitionScore([2, 2], 2)` = ?

**Bài 2 (tính tay).** `minPartitionScore([4, 1, 1, 4], 2)` — liệt kê ba cách cắt.

**Bài 3 (điền TODO).** Viết `min_with_cost(nums, k, g)` dùng DP thẳng cho hàm chi phí `g(s)` bất kỳ.

**Bài 4 (tự viết lại).** Viết lại Cách 3 với chi phí `s²` (biến thể 2).

**Bài 5 (mini challenge).** Trả về **các vị trí cắt** của cách chia tối ưu (truy vết từ CHT: lưu chỉ số `j` của đường trong deque).

In [23]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def min_with_cost(nums, k, g):
    # TODO: như dp_quadratic nhưng chi phí g(P[i] - P[j])
    pass

In [24]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def optimal_cuts(nums, k):
    # TODO: deque lưu thêm chỉ số j; arg[t][i] = j của đường đầu deque khi truy vấn i
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chỉ một cách `[2] [2]`. (2) `3 + 3`. (3) `6`.

**Bài 2.** (1) `[4] [1,1,4]`: `10 + 21 = 31`. (2) `[4,1] [1,4]`: `15 + 15 = 30`. (3) `[4,1,1] [4]`: `21 + 10 = 31` ⇒ đáp án `30`.

**Bài 3.** (1) Thay `s(s+1)/2` bằng `g(s)`. (2) Giữ ràng buộc đoạn khác rỗng. (3) Đối chiếu với `g = lambda s: s*(s+1)//2`.

**Bài 4.** (1) `(x − y)² = x² − 2xy + y²`. (2) Đường: `m = −2P_j`, `b = f + P_j²`. (3) Cộng `x²`.

**Bài 5.** (1) Mỗi đường mang theo `j`. (2) Ghi `arg[t][i]`. (3) Lần ngược từ `(k, n)`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2
assert Solution().minPartitionScore([2, 2], 2) == 6
assert Solution().minPartitionScore([4, 1, 1, 4], 2) == 30 == brute([4, 1, 1, 4], 2)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [26]:
# Bài 3
def min_with_cost_sol(nums, k, g):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1)
        for i in range(t, n + 1):
            cur[i] = min(prev[j] + g(P[i] - P[j]) for j in range(t - 1, i))
        prev = cur
    return prev[n]

assert min_with_cost_sol([5, 1, 2, 1], 2, lambda s: s * (s + 1) // 2) == 25
print("Bài 3 OK; với g(s) = s^3:", min_with_cost_sol([5, 1, 2, 1], 2, lambda s: s ** 3))

Bài 3 OK; với g(s) = s^3: 189


In [27]:
# Bài 4: chi phí s^2
def square_cost_cht(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); cht = MonotoneCHT()
        for i in range(t, n + 1):
            j = i - 1
            if prev[j] < INF: cht.add(-2 * P[j], prev[j] + P[j] * P[j])
            cur[i] = P[i] * P[i] + cht.query(P[i])
        prev = cur
    return prev[n]

rng = random.Random(4)
for _ in range(200):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, len(nums))
    assert square_cost_cht(nums, k) == min_with_cost_sol(nums, k, lambda s: s * s)
print("Bài 4 OK")

Bài 4 OK


In [28]:
# Bài 5: truy vết vị trí cắt
def optimal_cuts_sol(nums, k):
    n = len(nums); P = [0]
    for x in nums: P.append(P[-1] + x)
    prev = [0] + [INF] * n; arg = [[0] * (n + 1) for _ in range(k + 1)]
    for t in range(1, k + 1):
        cur = [INF] * (n + 1); dq = deque()          # phần tử (m, b, j)
        for i in range(t, n + 1):
            j = i - 1
            if prev[j] < INF:
                m, b = -P[j], prev[j] + (P[j] ** 2 - P[j]) // 2
                while len(dq) >= 2 and MonotoneCHT.bad(dq[-2][:2], dq[-1][:2], (m, b)): dq.pop()
                dq.append((m, b, j))
            x = P[i]
            while len(dq) >= 2 and dq[1][0] * x + dq[1][1] <= dq[0][0] * x + dq[0][1]: dq.popleft()
            cur[i] = (x * x + x) // 2 + dq[0][0] * x + dq[0][1]; arg[t][i] = dq[0][2]
        prev = cur
    cuts, i = [], n
    for t in range(k, 0, -1):
        j = arg[t][i]; cuts.append((j, i)); i = j
    return prev[n], cuts[::-1]

val, segs = optimal_cuts_sol([5, 1, 2, 1], 2)
print("Bài 5:", val, [[5, 1, 2, 1][a:b] for a, b in segs])
rng = random.Random(5)
for _ in range(200):
    nums = [rng.randint(1, 20) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, len(nums))
    v, sg = optimal_cuts_sol(nums, k)
    assert v == sum(sum(nums[a:b]) * (sum(nums[a:b]) + 1) // 2 for a, b in sg) == Solution().minPartitionScore(nums, k)
print("Bài 5 OK")

Bài 5: 25 [[5], [1, 2, 1]]
Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
P = tiền tố; prev = [0] + [inf]*n
for t in 1..k:
    deque rỗng
    for i in t..n:
        j = i - 1
        if prev[j] < inf: add line (m=-P[j], b=prev[j] + (P[j]^2 - P[j])//2)   # m giảm dần
        cur[i] = (P[i]^2 + P[i])//2 + query(P[i])                             # x tăng dần
    prev = cur
return prev[n]
```

**Checklist:**

- [ ] Đoạn khác rỗng: chỉ thêm `j ≤ i − 1` trước khi truy vấn `i`?
- [ ] Bỏ đường thừa bằng **nhân chéo số nguyên**, đúng chiều bất đẳng thức?
- [ ] Cộng lại phần `(P_i² + P_i)/2`?
- [ ] Chỉ thêm đường khi `prev[j]` hữu hạn?
- [ ] Đã đối chiếu với DP `O(k n²)` trên dữ liệu nhỏ?